# 秋田県クマ出没 — ExtraTrees ベンチマーク（全グリッド 260 セル版）

**入力**: `050008_kumadas.csv`（秋田県オープンデータ, CC-BY 4.0）  
**グリッド**: 13col × 20row = **260 セル** @ 10km × 10km  
**訓練**: 2022-2024（月次予測）　**テスト**: 2025（365 日）  

| 評価セクション | 対象 | 指標 |
|---------------|------|------|
| Sec 7 | **260 セル グローバル** | P@K / R@K (K=10,20,30) |
| **Sec 8** | **260 セル per-cell** | Brier / ECE / MAE / ROC-AUC（訂正版） |
| Sec 9 | TOP20 セル per-cell | TTM-512 との比較（参考） |

## 0) 依存ライブラリのインストール

In [ ]:
!pip install -q imbalanced-learn japanize_matplotlib
import imblearn, sklearn, matplotlib
print(f'imbalanced-learn: {imblearn.__version__}')
print(f'scikit-learn    : {sklearn.__version__}')
print(f'matplotlib      : {matplotlib.__version__}')

## 1) パラメータ設定

In [ ]:
import os, math, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# ── グリッドパラメータ（Yamagata_10km_Grid_0.csv と整合）────────────────────
LAT_STEP     = 0.090090
LON_STEP     = 0.114326
N_ROWS       = 20
N_COLS       = 13
N_CELLS      = N_ROWS * N_COLS   # 260
GRID_LAT_MIN = 38.839510
GRID_LON_MIN = 139.663417
GRID_LAT_MAX = GRID_LAT_MIN + N_ROWS * LAT_STEP
GRID_LON_MAX = GRID_LON_MIN + N_COLS * LON_STEP
REF_LAT      = 39.7
DEM_ZOOM     = 11

# ── 訓練・テスト ─────────────────────────────────────────────────────────────
TRAIN_YEARS  = [2022, 2023, 2024]
TEST_YEAR    = 2025
K_VALUES     = [10, 20, 30]

SEASONS = {
    'spring': [4, 5],
    'summer': [6, 7, 8],
    'fall':   [9, 10, 11],
    'winter': [12, 1, 2, 3],
}

# TOP20 セル（TTM-512 との比較用, Sec 9）
TOP20 = ['4_9','6_15','9_14','4_10','5_9','7_8','8_15','7_15',
         '3_11','3_10','6_14','9_15','5_8','5_15','9_16','6_7',
         '7_16','4_8','3_14','5_14']

# DEM キャッシュ
CACHE_DIR = '/content/.dem_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

print(f'グリッド: {N_ROWS}x{N_COLS}={N_CELLS} cells')
print(f'訓練: {TRAIN_YEARS}  テスト: {TEST_YEAR}')
print(f'lat: {GRID_LAT_MIN:.4f} ~ {GRID_LAT_MAX:.4f}')
print(f'lon: {GRID_LON_MIN:.4f} ~ {GRID_LON_MAX:.4f}')

## 2) データのアップロード

In [ ]:
from google.colab import files

print('050008_kumadas.csv をアップロードしてください')
uploaded = files.upload()
KUMADAS_PATH = list(uploaded.keys())[0]
print(f'アップロード完了: {KUMADAS_PATH}')

## 3) 出没データ読込 & グリッド割り当て

In [ ]:
def build_grid():
    rows = []
    for r in range(N_ROWS):
        for c in range(N_COLS):
            lat0 = GRID_LAT_MIN + r * LAT_STEP
            lon0 = GRID_LON_MIN + c * LON_STEP
            rows.append({'cell_id': f'{c}_{r}', 'row': r, 'col': c,
                         'lat_center': lat0 + LAT_STEP/2,
                         'lon_center': lon0 + LON_STEP/2,
                         'lat_min': lat0, 'lat_max': lat0 + LAT_STEP,
                         'lon_min': lon0, 'lon_max': lon0 + LON_STEP})
    return pd.DataFrame(rows)

def load_and_assign(csv_path, years=None):
    df = pd.read_csv(csv_path, encoding='utf-8-sig')
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    df = df[df['獣種'] == 'ツキノワグマ'].copy()
    df['event_date'] = pd.to_datetime(df['目撃日時'], format='%Y/%m/%d %H:%M', errors='coerce')
    df['latitude']   = pd.to_numeric(df['x(緯度)'],  errors='coerce')
    df['longitude']  = pd.to_numeric(df['y(経度)'],  errors='coerce')
    df = df.dropna(subset=['event_date', 'latitude', 'longitude'])
    df['year']  = df['event_date'].dt.year
    df['month'] = df['event_date'].dt.month
    if years:
        df = df[df['year'].isin(years)].copy()
    # グリッド割り当て
    df['col'] = np.floor((df['longitude'] - GRID_LON_MIN) / LON_STEP).astype(int)
    df['row'] = np.floor((df['latitude']  - GRID_LAT_MIN) / LAT_STEP).astype(int)
    in_grid   = df['col'].between(0, N_COLS-1) & df['row'].between(0, N_ROWS-1)
    n_out = (~in_grid).sum()
    if n_out > 0:
        print(f'  グリッド外: {n_out} 件 → 除外')
    df = df[in_grid].copy()
    df['cell_id']   = df['col'].astype(str) + '_' + df['row'].astype(str)
    df['date_only'] = df['event_date'].dt.normalize()
    return df

grid     = build_grid()
train_s  = load_and_assign(KUMADAS_PATH, TRAIN_YEARS)
test_s   = load_and_assign(KUMADAS_PATH, [TEST_YEAR])
all_s    = load_and_assign(KUMADAS_PATH)

print(f'\nグリッド     : {len(grid)} セル')
print(f'訓練 {TRAIN_YEARS}: {len(train_s):,} 件')
print(f'テスト {TEST_YEAR}: {len(test_s):,} 件')
print(f'2025 出没セル: {test_s.cell_id.nunique()} / {N_CELLS}')
print(f'年別件数     : {all_s.year.value_counts().sort_index().to_dict()}')

## 4) DEM（数値標高モデル）ダウンロード

国土地理院タイル API から 10km グリッドをカバーする標高データを取得します。  
キャッシュあり → 再実行時は即座に完了。

In [ ]:
import requests

def _deg2tile(lat, lon, z):
    n = 2 ** z
    x = int((lon + 180) / 360 * n)
    y = int((1 - math.log(math.tan(math.radians(lat)) +
             1 / math.cos(math.radians(lat))) / math.pi) / 2 * n)
    return x, y

def _tile2bbox(x, y, z):
    n = 2 ** z
    def merc(yy): return math.degrees(math.atan(math.sinh(math.pi * (1 - 2*yy/n))))
    return merc(y), merc(y+1), (x/n)*360-180, ((x+1)/n)*360-180

def _fetch_tile(z, x, y):
    cache = os.path.join(CACHE_DIR, f'dem_{z}_{x}_{y}.npy')
    if os.path.exists(cache):
        return np.load(cache)
    url = f'https://cyberjapandata.gsi.go.jp/xyz/dem/{z}/{x}/{y}.txt'
    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        rows = [[float(v) if v != 'e' else np.nan for v in line.split(',')]
                for line in r.text.strip().split('\n')]
        data = np.array(rows, dtype=np.float32)
    except Exception:
        data = np.full((256, 256), np.nan, dtype=np.float32)
    np.save(cache, data)
    return data

def download_dem():
    import pickle
    cache_file = os.path.join(CACHE_DIR, 'dem_assembled_z11_akita10km.pkl')
    if os.path.exists(cache_file):
        print('  キャッシュから DEM を読み込み...')
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    x_min, y_min = _deg2tile(GRID_LAT_MAX + 0.1, GRID_LON_MIN - 0.1, DEM_ZOOM)
    x_max, y_max = _deg2tile(GRID_LAT_MIN - 0.1, GRID_LON_MAX + 0.1, DEM_ZOOM)
    nx, ny = x_max - x_min + 1, y_max - y_min + 1
    print(f'  DEM ダウンロード: {nx}x{ny}={nx*ny} タイル...')
    assembled = np.full((ny*256, nx*256), np.nan, dtype=np.float32)
    done = 0
    for xi in range(x_min, x_max+1):
        for yi in range(y_min, y_max+1):
            tile = _fetch_tile(DEM_ZOOM, xi, yi)
            assembled[(yi-y_min)*256:(yi-y_min+1)*256,
                      (xi-x_min)*256:(xi-x_min+1)*256] = tile
            done += 1
            if done % 30 == 0:
                print(f'    {done}/{nx*ny}...')
            time.sleep(0.03)
    _, lat_max_r, lon_min_r, _ = _tile2bbox(x_min, y_min, DEM_ZOOM)
    lat_min_r, _, _, lon_max_r = _tile2bbox(x_max, y_max, DEM_ZOOM)
    info = {'data': assembled,
            'lat_min': lat_min_r, 'lat_max': lat_max_r,
            'lon_min': lon_min_r, 'lon_max': lon_max_r}
    with open(cache_file, 'wb') as f:
        pickle.dump(info, f)
    return info

print('DEM 関数定義 OK')

In [ ]:
dem = download_dem()
print(f'DEM 形状: {dem["data"].shape}')
print(f'lat: {dem["lat_min"]:.4f} ~ {dem["lat_max"]:.4f}')
print(f'lon: {dem["lon_min"]:.4f} ~ {dem["lon_max"]:.4f}')

## 5) 静的特徴量（標高・土地被覆・人口）の計算

In [ ]:
_LULC_COLS = ['lulc_water','lulc_paddy','lulc_crop','lulc_urban',
              'lulc_grass','lulc_deciduous','lulc_mixed','lulc_conifer','lulc_bare']

def elevation_features(grid_df, dem):
    arr  = dem['data']
    nrow, ncol = arr.shape
    dlat = dem['lat_max'] - dem['lat_min']
    dlon = dem['lon_max'] - dem['lon_min']
    recs = []
    for _, cell in grid_df.iterrows():
        r0 = int((dem['lat_max'] - cell['lat_max']) / dlat * nrow)
        r1 = int((dem['lat_max'] - cell['lat_min']) / dlat * nrow)
        c0 = int((cell['lon_min'] - dem['lon_min']) / dlon * ncol)
        c1 = int((cell['lon_max'] - dem['lon_min']) / dlon * ncol)
        r0 = max(0, min(r0, nrow-1)); r1 = max(r0+1, min(r1, nrow))
        c0 = max(0, min(c0, ncol-1)); c1 = max(c0+1, min(c1, ncol))
        px = arr[r0:r1, c0:c1].ravel()
        v  = px[~np.isnan(px)]
        recs.append({'cell_id':   cell['cell_id'],
                     'elev_mean': float(np.mean(v)) if len(v) else np.nan,
                     'elev_std':  float(np.std(v))  if len(v) else np.nan,
                     'elev_max':  float(np.max(v))  if len(v) else np.nan,
                     'elev_min':  float(np.min(v))  if len(v) else np.nan})
    return pd.DataFrame(recs)

def lulc_features(grid_df, elev):
    df = grid_df[['cell_id']].merge(elev[['cell_id','elev_mean']], on='cell_id', how='left')
    def classify(e):
        z = {c: 0.0 for c in _LULC_COLS}
        if pd.isna(e): return z
        if e < 0:      return {**z, 'lulc_water': 1.0}
        if e < 20:     return {**z, 'lulc_paddy': 0.55, 'lulc_crop': 0.20,
                               'lulc_urban': 0.15, 'lulc_grass': 0.10}
        if e < 100:    return {**z, 'lulc_crop': 0.40, 'lulc_urban': 0.20,
                               'lulc_grass': 0.20, 'lulc_deciduous': 0.20}
        if e < 400:    return {**z, 'lulc_deciduous': 0.60, 'lulc_mixed': 0.25,
                               'lulc_grass': 0.10, 'lulc_crop': 0.05}
        if e < 800:    return {**z, 'lulc_mixed': 0.50, 'lulc_deciduous': 0.30,
                               'lulc_conifer': 0.20}
        if e < 1500:   return {**z, 'lulc_conifer': 0.60, 'lulc_mixed': 0.30,
                               'lulc_bare': 0.10}
        return             {**z, 'lulc_bare': 0.55, 'lulc_conifer': 0.25, 'lulc_grass': 0.20}
    lulc = df['elev_mean'].apply(classify).apply(pd.Series)
    lulc.insert(0, 'cell_id', df['cell_id'].values)
    return lulc

_AKITA_CITIES = [
    (39.7192, 140.1025, 301000), (39.3147, 140.5618, 87000),
    (39.4625, 140.4849,  84000), (39.3906, 140.0490, 76000),
    (40.2797, 140.5640,  61000), (40.2092, 140.0313, 52000),
    (39.1621, 140.4964,  44000), (39.8883, 140.0189, 33000),
    (40.2231, 140.3773,  29000), (39.8806, 139.8494, 27000),
]

def pop_features(grid_df):
    lats = grid_df['lat_center'].values
    lons = grid_df['lon_center'].values
    pop_den  = np.zeros(len(grid_df), np.float32)
    min_dist = np.full(len(grid_df), np.inf, np.float32)
    for clat, clon, pop in _AKITA_CITIES:
        dy = (lats - clat) * 111.0
        dx = (lons - clon) * 111.0 * math.cos(math.radians(REF_LAT))
        d  = np.clip(np.sqrt(dy**2 + dx**2), 0.5, None)
        pop_den  += pop / d**2
        min_dist  = np.minimum(min_dist, d)
    scale = sum(p for _, _, p in _AKITA_CITIES) / 10.0
    return pd.DataFrame({'cell_id':          grid_df['cell_id'],
                         'pop_density':       (pop_den / scale).astype(np.float32),
                         'log_pop_density':   np.log1p(pop_den / scale).astype(np.float32),
                         'dist_nearest_city': min_dist})

elev   = elevation_features(grid, dem)
lulc   = lulc_features(grid, elev)
pop    = pop_features(grid)
static = (grid[['cell_id']]
          .merge(elev, on='cell_id', how='left')
          .merge(lulc, on='cell_id', how='left')
          .merge(pop,  on='cell_id', how='left'))

print(f'静的特徴量: {static.shape}  (cells x features)')
print(f'標高範囲: {elev.elev_mean.min():.0f} ~ {elev.elev_mean.max():.0f} m')
print(f'欠損数 : {static.isnull().sum().sum()}')

## 6) 訓練特徴量行列の構築 & ExtraTrees 学習

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from imblearn.under_sampling import RandomUnderSampler

FEATURE_COLS = [
    'elev_mean','elev_std','elev_max','elev_min',
    *_LULC_COLS,
    'pop_density','log_pop_density','dist_nearest_city',
    'month_sin','month_cos','season','active_season',
    'pre_hibernation','post_hibernation','years_since_start',
    'hist_positive_rate',
]

def temporal_features(year, month):
    return {
        'month_sin':         math.sin(2 * math.pi * month / 12),
        'month_cos':         math.cos(2 * math.pi * month / 12),
        'season':            (month - 1) // 3,
        'active_season':     int(4 <= month <= 11),
        'pre_hibernation':   int(month in (9, 10, 11)),
        'post_hibernation':  int(month in (4, 5)),
        'years_since_start': year - min(TRAIN_YEARS),
    }

def make_train_labels(sightings_df):
    all_cells  = grid['cell_id'].tolist()
    rows = [{'cell_id': c, 'year': y, 'month': m, 'label': 0}
            for y in TRAIN_YEARS for c in all_cells for m in range(1, 13)]
    labels = pd.DataFrame(rows)
    hits = (sightings_df.groupby(['cell_id','year','month']).size()
            .reset_index(name='cnt').assign(label=1))
    labels = labels.merge(hits[['cell_id','year','month','label']],
                          on=['cell_id','year','month'], how='left', suffixes=('','_h'))
    labels['label'] = labels['label_h'].fillna(0).astype(int)
    return labels.drop(columns='label_h')

def build_features(label_df, static_df, hist_rate):
    df = label_df.merge(static_df, on='cell_id', how='left')
    df = df.merge(hist_rate.rename('hist_positive_rate'), on='cell_id', how='left')
    temporal = pd.DataFrame(
        [temporal_features(r.year, r.month) for r in df[['year','month']].itertuples()],
        index=df.index)
    return pd.concat([df, temporal], axis=1)

# ── 訓練 ──────────────────────────────────────────────────────────────────
print('[1/3] 月次ラベル生成...')
train_labels = make_train_labels(train_s)
hist_rate    = train_labels.groupby('cell_id')['label'].mean()
pos = train_labels.label.sum()
print(f'  {len(train_labels):,} cell×month, pos={pos} ({100*pos/len(train_labels):.1f}%)')

print('[2/3] 特徴量行列の構築...')
train_df = build_features(train_labels, static, hist_rate)
X_train  = train_df[FEATURE_COLS].fillna(train_df[FEATURE_COLS].median())
y_train  = train_df['label']
print(f'  Shape: {X_train.shape}')

print('[3/3] ExtraTreesClassifier 学習（RandomUnderSampling）...')
rus = RandomUnderSampler(random_state=42)
Xr, yr = rus.fit_resample(X_train, y_train)
print(f'  アンダーサンプリング後: {len(yr):,} samples')
clf = ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(Xr, yr)

fi = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print('\n特徴量重要度 TOP5:')
print(fi.head(5).to_string())
print('\n学習完了')

## 7) グローバル評価（P@K / R@K — 全 260 セル）

全 260 セルを対象に日次ランキングで P@K / R@K を計算します。

In [ ]:
from sklearn.metrics import roc_auc_score

# ── 月次予測確率の生成（全 260 セル × 12 ヶ月）────────────────────────────
all_cells   = grid['cell_id'].tolist()
monthly_rows = [{'cell_id': c, 'year': TEST_YEAR, 'month': m}
                for m in range(1, 13) for c in all_cells]
mdf = pd.DataFrame(monthly_rows)
mdf = mdf.merge(static, on='cell_id', how='left')
mdf = mdf.merge(hist_rate.rename('hist_positive_rate'), on='cell_id', how='left')
temporal_m = pd.DataFrame(
    [temporal_features(r.year, r.month) for r in mdf[['year','month']].itertuples()],
    index=mdf.index)
mdf = pd.concat([mdf, temporal_m], axis=1)
Xm = mdf[FEATURE_COLS].fillna(mdf[FEATURE_COLS].median())
mdf['proba'] = clf.predict_proba(Xm)[:, 1]

proba_by_month = {mo: mdf[mdf.month == mo].set_index('cell_id')['proba']
                  for mo in range(1, 13)}

# ── 日次出没セットの構築 ────────────────────────────────────────────────
all_days = pd.date_range(f'{TEST_YEAR}-01-01', f'{TEST_YEAR}-12-31')
day_sightings = {}
for day in all_days:
    mask = test_s['event_date'].dt.date == day.date()
    day_sightings[day] = set(test_s[mask]['cell_id'].values)

avg_daily_pos = np.mean([len(v) for v in day_sightings.values()])
rnd_prec      = avg_daily_pos / N_CELLS

# ── P@K / R@K ──────────────────────────────────────────────────────────
day_p = {k: [] for k in K_VALUES}
day_r = {k: [] for k in K_VALUES}
for day in all_days:
    ranked  = proba_by_month[day.month].sort_values(ascending=False).index.tolist()
    present = day_sightings[day]
    n_pos   = len(present)
    for k in K_VALUES:
        hits = len(set(ranked[:k]) & present)
        day_p[k].append(hits / k)
        day_r[k].append(hits / n_pos if n_pos > 0 else np.nan)

# ── 全活動セルの月次 ROC-AUC ────────────────────────────────────────────
test_monthly = (test_s.groupby(['cell_id','month']).size()
                .reset_index(name='cnt').assign(label=1))
cell_roc = {}
for cid in all_cells:
    yt = []
    yp = []
    for mo in range(1, 13):
        row = test_monthly[(test_monthly.cell_id == cid) & (test_monthly.month == mo)]
        yt.append(1 if len(row) > 0 else 0)
        yp.append(float(proba_by_month[mo].get(cid, 0.5)))
    if 0 < sum(yt) < 12:
        try:
            cell_roc[cid] = roc_auc_score(yt, yp)
        except Exception:
            pass

# ── 出力 ──────────────────────────────────────────────────────────────
print('=' * 74)
print('AKITA — ExtraTrees 10km×10km  グローバル評価')
print(f'Grid: {N_ROWS}x{N_COLS}={N_CELLS} cells | Test: {TEST_YEAR} (365 days)')
print(f'Train: {TRAIN_YEARS} | 平均出没セル/日: {avg_daily_pos:.1f} | ランダムP: {rnd_prec:.4f}')
print('=' * 74)
print(f'  月次 ROC-AUC (全活動セル): {np.mean(list(cell_roc.values())):.3f}  '
      f'(n={len(cell_roc)})')
print()
print(f'  {"":4} {"P@K":>8} {"vs rnd":>8} {"R@K":>8}')
print('  ' + '-'*32)
for k in K_VALUES:
    rv    = [v for v in day_r[k] if not np.isnan(v)]
    p_avg = np.mean(day_p[k])
    r_avg = np.mean(rv) if rv else 0.0
    print(f'  K={k:2d}  {p_avg:8.4f}  {p_avg/rnd_prec:6.1f}x  {r_avg:8.4f}')

print()
print('  季節別 K=20:')
for season, months in SEASONS.items():
    ps, rs = [], []
    for day in all_days:
        if day.month not in months:
            continue
        ranked  = proba_by_month[day.month].sort_values(ascending=False).index.tolist()
        present = day_sightings[day]
        n_pos   = len(present)
        hits    = len(set(ranked[:20]) & present)
        ps.append(hits / 20)
        if n_pos > 0:
            rs.append(hits / n_pos)
    print(f'  {season:<10} P@20={np.mean(ps):.3f}  R@20={np.mean(rs):.3f}')

## 8) 全グリッド per-cell キャリブレーション指標（260 セル版・訂正値）

従来 TOP20 セルのみで報告されていた Brier / ECE / MAE を **260 セル全体**で再計算します。  
評価可能セル = 2025 年に 1 件以上の出没があったセル（AUC 計算可能な条件）。

In [ ]:
from sklearn.metrics import average_precision_score, brier_score_loss

def ece_score(yt, yp, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    n    = len(yt)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (yp >= lo) & (yp < hi)
        if not mask.any():
            continue
        ece += mask.sum()/n * abs(yt[mask].mean() - yp[mask].mean())
    return float(ece)

def compute_percell_metrics(cell_list, proba_by_month, day_sightings):
    """任意のセルリストについて日次 per-cell 指標を計算する汎用関数。"""
    all_days_sorted = sorted(day_sightings.keys())
    n_days = len(all_days_sorted)

    day_rankings = {
        d: proba_by_month[d.month].sort_values(ascending=False).index.tolist()
        for d in all_days_sorted
    }

    records = []
    for cid in cell_list:
        yp    = np.array([float(proba_by_month[d.month].get(cid, 0.0))
                          for d in all_days_sorted])
        yt    = np.array([int(cid in day_sightings[d]) for d in all_days_sorted])
        n_pos = int(yt.sum())
        if n_pos == 0 or n_pos == n_days:
            continue
        try:
            roc  = roc_auc_score(yt, yp)
            pr   = average_precision_score(yt, yp)
        except Exception:
            roc = pr = np.nan
        brier = brier_score_loss(yt, yp)
        mae   = float(np.mean(np.abs(yp - yt)))
        rmse  = float(np.sqrt(np.mean((yp - yt)**2)))
        ece   = ece_score(yt, yp)

        rec = {'grid_id': cid,
               'PR_AUC': pr, 'ROC_AUC': roc,
               'Brier': brier, 'ECE': ece, 'MAE': mae, 'RMSE': rmse,
               'n_pos': n_pos}
        for k in K_VALUES:
            in_topk = np.array([int(cid in day_rankings[d][:k])
                                 for d in all_days_sorted])
            tp  = int((in_topk & yt.astype(bool)).sum())
            p_k = tp / max(in_topk.sum(), 1)
            r_k = tp / max(n_pos, 1)
            rec[f'Precision_{k}'] = p_k
            rec[f'Recall_{k}']    = r_k
            if k == 10:
                rec['Hit_Top10'] = r_k
        records.append(rec)

    return pd.DataFrame(records).set_index('grid_id')

print('compute_percell_metrics 関数定義 OK')

In [ ]:
print('全グリッド 260 セルで per-cell 指標を計算中...')
et_allgrid = compute_percell_metrics(all_cells, proba_by_month, day_sightings)

metric_cols = ['PR_AUC','ROC_AUC','Brier','ECE','MAE','RMSE']
m_all = et_allgrid.mean()

print()
print('=' * 60)
print(f'全グリッド ({len(et_allgrid)} セル) per-cell 平均指標  [Test={TEST_YEAR}]')
print(f'（評価可能条件: 2025 年に 1 件以上の出没があったセル）')
print('=' * 60)

# 旧TOP20値（参考）
OLD_TOP20 = {'Brier': 0.564, 'ECE': 0.176, 'MAE': 0.613}

fmt = '  {:<12}: {:>8.4f}  {}'
for col in metric_cols:
    val = float(m_all[col])
    note = ''
    if col in OLD_TOP20:
        note = f'  (旧TOP20報告値={OLD_TOP20[col]:.3f} → 訂正)'
    print(fmt.format(col, val, note))

print()
print(f'  評価可能セル: {len(et_allgrid)} / {N_CELLS}')
print(f'  (2025年出没ゼロセル {N_CELLS - len(et_allgrid)} 個は AUC 計算不可 → 除外)')

## 9) TOP20 per-cell 指標（TTM-512 との比較用・参考）

TTM-512 のセルごと評価は TOP20 セルのみ実施済みのため、ET も同じ 20 セルで比較します。

In [ ]:
et_top20 = compute_percell_metrics(TOP20, proba_by_month, day_sightings)

TTM_MEAN = {
    'PR_AUC':0.3451,'ROC_AUC':0.5465,'Brier':0.2535,'ECE':0.2098,
    'MAE':0.3410,'RMSE':0.5016,'Hit_Top10':0.1075,
    'Precision_10':0.3222,'Precision_20':0.3274,'Precision_30':0.3202,
    'Recall_10':0.1075,'Recall_20':0.2181,'Recall_30':0.3174,
}
et_mean = et_top20.mean()
metric_list = ['PR_AUC','ROC_AUC','Brier','ECE','MAE','RMSE','Hit_Top10',
               'Precision_10','Precision_20','Precision_30',
               'Recall_10','Recall_20','Recall_30']
better_hi = {'PR_AUC','ROC_AUC','Hit_Top10',
             'Precision_10','Precision_20','Precision_30',
             'Recall_10','Recall_20','Recall_30'}

print('=' * 80)
print(f'TOP20 セル 平均比較: ExtraTrees vs TTM-512  (n={len(et_top20)}, Test={TEST_YEAR})')
print('=' * 80)
fmt = '  {:<22} {:>12} {:>12} {:>12} {:>8}'
print(fmt.format('指標','ExtraTrees','TTM-512','差(ET-TTM)','優位'))
print('  ' + '-'*72)
for m in metric_list:
    ev   = float(et_mean.get(m, float('nan')))
    tv   = TTM_MEAN.get(m, float('nan'))
    diff = ev - tv
    if m in better_hi:
        better = 'ET↑' if diff > 0 else 'TTM↑'
    else:
        better = 'ET↑' if diff < 0 else 'TTM↑'
    print(fmt.format(m, f'{ev:.4f}', f'{tv:.4f}', f'{diff:+.4f}', better))

TTM_ROC = {'5_14':0.6352,'4_8':0.5838,'6_7':0.5545,'5_15':0.5000,'3_14':0.6690,
           '7_16':0.6549,'9_16':0.7172,'5_8':0.4313,'6_14':0.5675,'8_15':0.5224,
           '9_14':0.6928,'3_11':0.4206,'9_15':0.6878,'6_15':0.5530,'7_15':0.3565,
           '3_10':0.5055,'5_9':0.4630,'4_10':0.5541,'7_8':0.5358,'4_9':0.3257}
print()
print('  【TOP20 per-cell ROC-AUC】')
print(f'  {"cell":>8} {"ET":>8} {"TTM":>8} {"diff":>8}')
for cid in TOP20:
    ev = float(et_top20.loc[cid,'ROC_AUC']) if cid in et_top20.index else float('nan')
    tv = TTM_ROC.get(cid, float('nan'))
    print(f'  {cid:>8} {ev:>8.3f} {tv:>8.3f} {ev-tv:>+8.3f}')

## 10) 可視化

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib

# グローバル P@K / R@K
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f'Akita ET 10km — Global P@K / R@K  (Test {TEST_YEAR}, {N_CELLS} cells)',
             fontweight='bold')
x = np.arange(len(K_VALUES)); w = 0.35
p_vals = [np.mean(day_p[k]) for k in K_VALUES]
r_raw  = [[v for v in day_r[k] if not np.isnan(v)] for k in K_VALUES]
r_vals = [np.mean(rv) if rv else 0.0 for rv in r_raw]
rnd_r  = [k/N_CELLS for k in K_VALUES]

axes[0].bar(x-w/2, p_vals, w, label='ExtraTrees', color='steelblue', alpha=0.85)
axes[0].bar(x+w/2, [rnd_prec]*3, w, label='Random', color='gray', alpha=0.55)
axes[0].axhline(rnd_prec, color='red', ls='--', lw=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels([f'K={k}' for k in K_VALUES])
axes[0].set_title('Precision@K'); axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x-w/2, r_vals, w, label='ExtraTrees', color='steelblue', alpha=0.85)
axes[1].bar(x+w/2, rnd_r, w, label='Random', color='gray', alpha=0.55)
axes[1].set_xticks(x); axes[1].set_xticklabels([f'K={k}' for k in K_VALUES])
axes[1].set_title('Recall@K'); axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
import japanize_matplotlib

# ROC-AUC ヒストグラム（260 セル）
roc_vals = et_allgrid['ROC_AUC'].dropna()
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(roc_vals, bins=20, color='steelblue', edgecolor='white')
ax.axvline(0.5, color='red', ls='--', label='random (0.5)')
ax.axvline(roc_vals.mean(), color='green', ls='--',
           label=f'mean ({roc_vals.mean():.3f})')
ax.set_title(f'セルごと ROC-AUC 分布（全グリッド {len(et_allgrid)} セル）')
ax.set_xlabel('ROC-AUC'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# 空間ヒートマップ（Brier）
heat_brier = np.full((N_ROWS, N_COLS), np.nan)
for cid, row in et_allgrid.iterrows():
    col_i, row_i = map(int, cid.split('_'))
    heat_brier[row_i, col_i] = row['Brier']

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(heat_brier, origin='lower', cmap='YlOrRd', aspect='auto',
               vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Brier Score')
ax.set_xlabel('Col（経度方向）'); ax.set_ylabel('Row（緯度方向）')
ax.set_title(f'秋田県 10km グリッド セル別 Brier Score（全グリッド版, {TEST_YEAR}年）')
ax.set_xticks(range(N_COLS)); ax.set_yticks(range(N_ROWS))
plt.tight_layout(); plt.show()

# TOP20 vs ALL-GRID 指標比較
compare_metrics = ['Brier','ECE','MAE','RMSE']
top20_vals  = [float(et_top20.mean()[m])   for m in compare_metrics]
all_vals    = [float(et_allgrid.mean()[m]) for m in compare_metrics]
old_vals    = [0.564, 0.176, 0.613, float('nan')]

x = np.arange(len(compare_metrics)); w = 0.28
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w, top20_vals, w, label=f'ET TOP20 (n={len(et_top20)})', color='steelblue', alpha=0.8)
ax.bar(x,     all_vals,   w, label=f'ET AllGrid (n={len(et_allgrid)})', color='darkorange', alpha=0.8)
valid_old = [(i, v) for i, v in enumerate(old_vals) if not (isinstance(v, float) and np.isnan(v))]
for i, v in valid_old:
    ax.bar(x[i] + w, v, w, color='gray', alpha=0.6,
           label='旧TOP20報告値' if i == valid_old[0][0] else '_')
ax.set_xticks(x); ax.set_xticklabels(compare_metrics)
ax.set_title('Akita ET — キャリブレーション指標 比較\n(TOP20 / AllGrid 260セル / 旧TOP20報告値)')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 11) CSV 保存 & ダウンロード

In [ ]:
from google.colab import files

# 全グリッド per-cell
et_allgrid.to_csv('akita_allgrid_percell_2025.csv', float_format='%.6f')
print(f'全グリッド per-cell: akita_allgrid_percell_2025.csv  ({len(et_allgrid)} rows)')

# TOP20 per-cell
et_top20.to_csv('akita_top20_percell_2025.csv', float_format='%.6f')
print(f'TOP20 per-cell    : akita_top20_percell_2025.csv  ({len(et_top20)} rows)')

# グローバル P@K サマリ
global_rows = []
for k in K_VALUES:
    rv    = [v for v in day_r[k] if not np.isnan(v)]
    global_rows.append({
        'K':           k,
        'Precision@K': round(np.mean(day_p[k]), 6),
        'Recall@K':    round(np.mean(rv) if rv else 0.0, 6),
        'mult_vs_rnd': round(np.mean(day_p[k]) / rnd_prec, 3),
        'n_cells_pool': N_CELLS,
        'avg_daily_pos': round(avg_daily_pos, 3),
        'rnd_P':        round(rnd_prec, 6),
    })
pd.DataFrame(global_rows).to_csv('akita_et_global_pkrk_2025.csv', index=False)
print(f'グローバル P@K    : akita_et_global_pkrk_2025.csv')

# 全グリッド平均指標サマリ（訂正値）
summary_rows = []
for col in ['PR_AUC','ROC_AUC','Brier','ECE','MAE','RMSE']:
    summary_rows.append({
        'metric': col,
        'AllGrid_260cells': round(float(et_allgrid.mean()[col]), 6),
        'TOP20_cells':      round(float(et_top20.mean().get(col, float('nan'))), 6),
        'n_allgrid': len(et_allgrid),
        'n_top20':   len(et_top20),
    })
pd.DataFrame(summary_rows).to_csv('akita_et_percell_summary_corrected.csv', index=False)
print(f'訂正値サマリ       : akita_et_percell_summary_corrected.csv')

print()
print('--- ダウンロード ---')
files.download('akita_allgrid_percell_2025.csv')
files.download('akita_top20_percell_2025.csv')
files.download('akita_et_global_pkrk_2025.csv')
files.download('akita_et_percell_summary_corrected.csv')
print('完了')